In [ ]:
# Step 0: Extract and load data
import os
import tarfile
import pandas as pd
import numpy as np

archive_path = 'data/hwu.tar.gz'
if os.path.exists(archive_path):
    try:
        with tarfile.open(archive_path, 'r:gz') as tar:
            tar.extractall(path='data')
        print('Archive extracted to data/')
    except Exception as e:
        print('Could not extract archive:', e)
else:
    print(f"Archive not found at {archive_path}. If you already extracted, ignore this message.")

def safe_read(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}. Make sure archive contains this file and it was extracted.")
    return pd.read_csv(path, sep='\t', header=None, names=['text', 'intent'])

train_path = 'hwu_train.csv'
val_path = 'hwu_val.csv'
test_path = 'hwu_test.csv'

for p in [train_path, val_path, test_path]:
    if not os.path.exists(p):
        # also check in data/ folder
        if os.path.exists(os.path.join('data', p)):
            p = os.path.join('data', p)
        else:
            print(f"Warning: {p} not found in working dir or data/. Please place the CSV files in the notebook folder or in data/.")

try:
    df_train = safe_read(train_path if os.path.exists(train_path) else os.path.join('data', train_path))
    df_val = safe_read(val_path if os.path.exists(val_path) else os.path.join('data', val_path))
    df_test = safe_read(test_path if os.path.exists(test_path) else os.path.join('data', test_path))
    print('Train shape:', df_train.shape)
    print('Val shape:', df_val.shape)
    print('Test shape:', df_test.shape)
except FileNotFoundError as e:
    print('Data files missing:', e)


In [ ]:
# Label encoding intents
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
all_intents = pd.concat([df_train['intent'], df_val['intent'], df_test['intent']]) if 'df_train' in globals() else None
if all_intents is None:
    print('Skipping label encoding because data not loaded')
else:
    le.fit(all_intents)
    y_train = le.transform(df_train['intent'])
    y_val = le.transform(df_val['intent'])
    y_test = le.transform(df_test['intent'])
    num_classes = len(le.classes_)
    print('Number of classes:', num_classes)


## Task 1: TF-IDF + Logistic Regression baseline

We build a scikit-learn pipeline, train on train set, evaluate on test set.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report, f1_score, log_loss

tfidf_lr_pipeline = make_pipeline(
    TfidfVectorizer(max_features=5000, ngram_range=(1,2)),
    LogisticRegression(max_iter=1000)
)
if 'df_train' in globals():
    tfidf_lr_pipeline.fit(df_train['text'], y_train)
    y_pred_tfidf = tfidf_lr_pipeline.predict(df_test['text'])
    try:
        y_proba_tfidf = tfidf_lr_pipeline.predict_proba(df_test['text'])
    except Exception:
        # if logistic doesn't support predict_proba for some reason, use decision_function then softmax
        import numpy as _np
        scores = tfidf_lr_pipeline.decision_function(df_test['text'])
        # softmax
        exp = _np.exp(scores - _np.max(scores, axis=1, keepdims=True))
        y_proba_tfidf = exp / _np.sum(exp, axis=1, keepdims=True)
    f1_tfidf = f1_score(y_test, y_pred_tfidf, average='macro')
    loss_tfidf = log_loss(y_test, y_proba_tfidf)
    print('TF-IDF + LR -- Macro F1:', f1_tfidf)
    print('TF-IDF + LR -- Test log loss:', loss_tfidf)
    print('\nClassification report:\n')
    print(classification_report(y_test, y_pred_tfidf, target_names=le.classes_))
else:
    print('Data not loaded; skipping TF-IDF pipeline')


## Task 2: Word2Vec (average) + Dense (Keras)
Train a Word2Vec on training texts, compute averaged sentence vectors, then train a simple Keras classifier.


In [ ]:
from gensim.models import Word2Vec
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

def tokenize_texts(texts):
    return [str(t).lower().split() for t in texts]

if 'df_train' in globals():
    sentences = tokenize_texts(df_train['text'])
    w2v_model = Word2Vec(sentences, vector_size=100, window=5, min_count=1, workers=4, seed=42)
    vector_size = w2v_model.vector_size
    print('Trained Word2Vec on training texts; vector size =', vector_size)

    def sentence_to_avg_vector(text, model, vector_size=vector_size):
        tokens = str(text).lower().split()
        vecs = [model.wv[w] for w in tokens if w in model.wv]
        if len(vecs) == 0:
            return np.zeros(vector_size, dtype=float)
        return np.mean(vecs, axis=0)

    X_train_avg = np.vstack([sentence_to_avg_vector(t, w2v_model) for t in df_train['text']])
    X_val_avg = np.vstack([sentence_to_avg_vector(t, w2v_model) for t in df_val['text']])
    X_test_avg = np.vstack([sentence_to_avg_vector(t, w2v_model) for t in df_test['text']])

    # Build simple dense classifier
    num_classes = len(le.classes_)
    model_avg = Sequential([
        Dense(128, activation='relu', input_shape=(vector_size,)),
        Dropout(0.5),
        Dense(num_classes, activation='softmax')
    ])
    model_avg.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    es = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
    history_avg = model_avg.fit(X_train_avg, y_train, validation_data=(X_val_avg, y_val), epochs=20, batch_size=32, callbacks=[es])
    loss_avg, acc_avg = model_avg.evaluate(X_test_avg, y_test, verbose=0)
    y_proba_avg = model_avg.predict(X_test_avg)
    y_pred_avg = np.argmax(y_proba_avg, axis=1)
    f1_avg = f1_score(y_test, y_pred_avg, average='macro')
    print('Word2Vec AVG + Dense -- Macro F1:', f1_avg)
    print('Word2Vec AVG + Dense -- Test loss:', loss_avg)
else:
    print('Data not available; skipping Word2Vec avg pipeline')


## Task 3: Pre-trained Word2Vec embedding + LSTM
We reuse the Word2Vec trained previously as 'pre-trained' weights for an Embedding layer.


In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Embedding, LSTM
from tensorflow.keras.optimizers import Adam

if 'df_train' in globals():
    # Tokenize across train + val + test to ensure consistent index
    tokenizer = Tokenizer(oov_token='<UNK>')
    tokenizer.fit_on_texts(pd.concat([df_train['text'], df_val['text'], df_test['text']]))
    train_seq = tokenizer.texts_to_sequences(df_train['text'])
    val_seq = tokenizer.texts_to_sequences(df_val['text'])
    test_seq = tokenizer.texts_to_sequences(df_test['text'])
    max_len = 50
    X_train_pad = pad_sequences(train_seq, maxlen=max_len, padding='post')
    X_val_pad = pad_sequences(val_seq, maxlen=max_len, padding='post')
    X_test_pad = pad_sequences(test_seq, maxlen=max_len, padding='post')

    vocab_size = min(len(tokenizer.word_index) + 1, 50000)
    embedding_dim = w2v_model.vector_size
    embedding_matrix = np.zeros((vocab_size, embedding_dim))
    for word, i in tokenizer.word_index.items():
        if i >= vocab_size:
            continue
        if word in w2v_model.wv:
            embedding_matrix[i] = w2v_model.wv[word]

    lstm_model_pretrained = Sequential([
        Embedding(input_dim=vocab_size, output_dim=embedding_dim, weights=[embedding_matrix], input_length=max_len, trainable=False),
        LSTM(128, dropout=0.2, recurrent_dropout=0.2),
        Dense(num_classes, activation='softmax')
    ])
    lstm_model_pretrained.compile(optimizer=Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    es = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
    history_pre = lstm_model_pretrained.fit(X_train_pad, y_train, validation_data=(X_val_pad, y_val), epochs=15, batch_size=64, callbacks=[es])
    loss_pre, acc_pre = lstm_model_pretrained.evaluate(X_test_pad, y_test, verbose=0)
    y_proba_pre = lstm_model_pretrained.predict(X_test_pad)
    y_pred_pre = np.argmax(y_proba_pre, axis=1)
    f1_pre = f1_score(y_test, y_pred_pre, average='macro')
    print('Pretrained Embedding + LSTM -- Macro F1:', f1_pre)
    print('Pretrained Embedding + LSTM -- Test loss:', loss_pre)
else:
    print('Data not available; skipping pretrained embedding LSTM')


## Task 4: From-scratch Embedding + LSTM
Train the embedding layer from scratch (trainable).


In [ ]:
if 'df_train' in globals():
    embedding_dim_scratch = 100
    lstm_model_scratch = Sequential([
        Embedding(input_dim=vocab_size, output_dim=embedding_dim_scratch, input_length=max_len),
        LSTM(128, dropout=0.2, recurrent_dropout=0.2),
        Dense(num_classes, activation='softmax')
    ])
    lstm_model_scratch.compile(optimizer=Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    es = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
    history_scratch = lstm_model_scratch.fit(X_train_pad, y_train, validation_data=(X_val_pad, y_val), epochs=15, batch_size=64, callbacks=[es])
    loss_scratch, acc_scratch = lstm_model_scratch.evaluate(X_test_pad, y_test, verbose=0)
    y_proba_scratch = lstm_model_scratch.predict(X_test_pad)
    y_pred_scratch = np.argmax(y_proba_scratch, axis=1)
    f1_scratch = f1_score(y_test, y_pred_scratch, average='macro')
    print('Scratch Embedding + LSTM -- Macro F1:', f1_scratch)
    print('Scratch Embedding + LSTM -- Test loss:', loss_scratch)
else:
    print('Data not available; skipping scratch embedding LSTM')


## Task 5: Aggregate results and qualitative analysis
We will collect macro F1 and test loss for each pipeline into a table. Then run a small qualitative comparison on example sentences.


In [ ]:
import pandas as _pd
results = []
if 'f1_tfidf' in globals():
    results.append({'Pipeline':'TF-IDF + LR','F1_macro':f1_tfidf,'Test_loss':loss_tfidf})
if 'f1_avg' in globals():
    results.append({'Pipeline':'Word2Vec AVG + Dense','F1_macro':f1_avg,'Test_loss':loss_avg})
if 'f1_pre' in globals():
    results.append({'Pipeline':'Pretrained Emb + LSTM','F1_macro':f1_pre,'Test_loss':loss_pre})
if 'f1_scratch' in globals():
    results.append({'Pipeline':'Scratch Emb + LSTM','F1_macro':f1_scratch,'Test_loss':loss_scratch})

results_df = _pd.DataFrame(results).sort_values('F1_macro', ascending=False)
if not results_df.empty:
    display(results_df)
else:
    print('No results to display (likely because data or training was skipped).')

# Qualitative examples
examples = [
    'can you remind me to not call my mom',
    'is it going to be sunny or rainy tomorrow',
    'find a flight from new york to london but not through paris'
]

def predict_all(text):
    out = {}
    if 'tfidf_lr_pipeline' in globals():
        try:
            y = tfidf_lr_pipeline.predict([text])[0]
            out['TFIDF+LR'] = le.classes_[y]
        except Exception as e:
            out['TFIDF+LR'] = f'error: {e}'
    if 'w2v_model' in globals():
        vec = sentence_to_avg_vector(text, w2v_model)
        if 'model_avg' in globals():
            yprob = model_avg.predict(vec.reshape(1,-1))
            out['W2V-AVG'] = le.classes_[np.argmax(yprob)]
    if 'tokenizer' in globals():
        seq = tokenizer.texts_to_sequences([text])
        pad = pad_sequences(seq, maxlen=max_len, padding='post')
        if 'lstm_model_pretrained' in globals():
            yprob = lstm_model_pretrained.predict(pad)
            out['Pretrained-LSTM'] = le.classes_[np.argmax(yprob)]
        if 'lstm_model_scratch' in globals():
            yprob = lstm_model_scratch.predict(pad)
            out['Scratch-LSTM'] = le.classes_[np.argmax(yprob)]
    return out

for ex in examples:
    print('\nExample:', ex)
    preds = predict_all(ex)
    for k,v in preds.items():
        print(f'  {k}: {v}')

print('\nDone qualitative examples.')


### Notes and suggestions
- You can further improve results by using pre-trained embeddings (fastText, GloVe) or transformers (BERT).  
- When datasets/classes are imbalanced, prefer macro-F1 as requested.  
- Monitor training curves to detect overfitting; consider lowering model size or adding more regularization if necessary.
